# 2 · Train a delphinID classifier

Trains a delphinID CNN on detection frames, tests it with grouped cross-validation, and exports a model that runs in PAMGuard's Deep Learning module.

**You need** a frames CSV and its `_params.json` from `01_make_frames` (or frames made another way; see the format at the end of notebook 1). By default this notebook uses a small **synthetic** example (`examples/frames_example_synthetic.csv`) so you can try it straight away.

**You get**, in `OUTPUT_DIR`:
- `cv_predictions.csv`: cross-validation predictions for every frame (standard classifier output format)
- `cv_history.csv`, `cv_summary.json`: training curves and accuracies
- `<MODEL_NAME>.zip`: the final model for PAMGuard

**How testing works.** Each group (by default each event) is left out in turn and predicted by a model trained on all the other groups, so accuracy reflects performance on events the model has never seen. Within each fold the model trains on `N_BOOTSTRAPS` successive random subsamples, each taking at most `MAX_PER_GROUP` frames per group: this stops a few long events dominating while still using most of the data. The final model is trained the same way on all the data.

## Setup
**Google Colab** (no local TensorFlow install needed): run the next cell, then upload your frames CSV and params JSON with the file browser on the left and set the paths below. A GPU isn't needed.

**Locally**: `pip install -e "python[delphinid,notebooks]"` from the repository folder, and skip the next cell.

In [ ]:
# Google Colab only
# %pip install -q "speciesclassifiers @ git+https://github.com/tristankleyn/speciesClassifiers#subdirectory=python"

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from speciesclassifiers.delphinid import (AcousticParams, CNNParams, ResamplingParams, TrainingParams,
                                          cross_validate, export_pamguard, fit_bootstrapped, summarise)
from speciesclassifiers.delphinid.plots import plot_confusion, plot_history

## Parameters

In [ ]:
# --- Data ----------------------------------------------------------------------------------
FRAMES_CSV  = "../../examples/frames_example_synthetic.csv"
PARAMS_JSON = "../../examples/frames_example_synthetic_params.json"   # from 01_make_frames
CLASSES     = None          # list of labels to use, e.g. ["Dde", "Ggr", "Ttr"]; None = all
GROUP_COL   = "event_id"    # independent unit for cross-validation (e.g. "event_id", "location")
OUTPUT_DIR  = "delphinID_run"
MODEL_NAME  = "delphinID_whistles"

# --- CNN (shape is fixed; sizes adjustable) ---------------------------------------------------
N_FILTERS   = 16      # filters per convolutional layer
KERNEL_SIZE = 3       # first conv layer kernel (second uses KERNEL_SIZE + 2)
MAX_POOL    = 2       # pooling window
LEAKY_RELU  = 0.1     # LeakyReLU slope for negative inputs
DENSE_SIZE  = 10      # hidden dense layer units
DROPOUT     = 0.3     # fraction of hidden units dropped while training
L2          = 1e-4    # L2 regularisation on the hidden dense layer

# --- Training ---------------------------------------------------------------------------------
LEARNING_RATE = 0.0005
EPOCHS        = 20     # per bootstrap
BATCH_SIZE    = 2
PATIENCE      = 20     # early stopping patience (epochs)
SEED          = 42

# --- Grouped resampling -----------------------------------------------------------------------
MAX_PER_GROUP = 30     # most frames taken from one group in each bootstrap
N_BOOTSTRAPS  = 5      # subsamples trained on in turn

# --- What to run ------------------------------------------------------------------------------
RUN_CROSS_VALIDATION = True
N_FOLDS     = None     # None = leave one group out at a time (most thorough, slowest);
                       # e.g. 10 = split groups into 10 folds (10 models instead of one per group)
TEST_GROUPS = None     # leave-one-group-out only: None = every group; or a short list for a quick trial

In [ ]:
cnn = CNNParams(N_FILTERS, KERNEL_SIZE, MAX_POOL, LEAKY_RELU, DENSE_SIZE, DROPOUT, L2)
training = TrainingParams(LEARNING_RATE, EPOCHS, BATCH_SIZE, PATIENCE, SEED)
resampling = ResamplingParams(MAX_PER_GROUP, N_BOOTSTRAPS)

meta = json.loads(Path(PARAMS_JSON).read_text())
acoustic = AcousticParams(**{k: v for k, v in meta["acoustic_params"].items() if k in AcousticParams.__dataclass_fields__})
sample_rate = meta.get("sample_rate")
out = Path(OUTPUT_DIR); out.mkdir(parents=True, exist_ok=True)
acoustic

## Load and check frames
Each label needs at least two groups (one to test, the rest to train). Labels with fewer are dropped.

In [ ]:
frames = pd.read_csv(FRAMES_CSV)
frames["label"] = frames["label"].astype(str)
if CLASSES is not None:
    frames = frames[frames["label"].isin(CLASSES)]
groups_per_label = frames.groupby("label")[GROUP_COL].nunique()
too_few = groups_per_label[groups_per_label < 2].index.tolist()
if too_few:
    print(f"Dropping labels with fewer than 2 groups: {too_few}")
    frames = frames[~frames["label"].isin(too_few)]
classes = sorted(frames["label"].unique())
n_inputs = sum(c[0] == "f" and c[1:].isdigit() for c in frames.columns)
expected = acoustic.n_bins(sample_rate)
assert n_inputs == expected, f"Frames have {n_inputs} features but the params give {expected}"
print(f"{len(frames)} frames, {frames[GROUP_COL].nunique()} groups, {len(classes)} classes, {n_inputs} inputs")
frames.groupby("label").agg(groups=(GROUP_COL, "nunique"), frames=(GROUP_COL, "size"))

## Cross-validate
One model is trained per fold, and each takes `N_BOOTSTRAPS × EPOCHS` epochs. With the defaults, a fold takes around 20 s on a laptop CPU for a few hundred frames per bootstrap, and longer with more data. Leave-one-group-out on hundreds of groups can take many hours: use `N_FOLDS`, or `TEST_GROUPS` for a quick trial, or run in Colab and leave it going.

In [ ]:
if RUN_CROSS_VALIDATION:
    preds, history = cross_validate(frames, classes, label_col="label", group_col=GROUP_COL, cnn=cnn,
                                    training=training, resampling=resampling, test_groups=TEST_GROUPS, n_folds=N_FOLDS,
                                    classifier=MODEL_NAME, voc_type=acoustic.voc_type)
    preds.to_csv(out / "cv_predictions.csv", index=False)
    history.to_csv(out / "cv_history.csv", index=False)

## Results
Frame accuracy: share of single frames classified correctly. Event accuracy: share of events whose summed frame probabilities pick the right class.

In [ ]:
if RUN_CROSS_VALIDATION:
    s = summarise(preds)
    print(f"Frame accuracy: {s['frame_accuracy']:.1%}   Event accuracy: {s['event_accuracy']:.1%}")
    display(s["event_accuracy_by_class"].rename("event accuracy").to_frame().style.format("{:.0%}"))
    fig, ax = plt.subplots(1, 2, figsize=(4 + 1.6 * len(classes), 1.5 + 0.7 * len(classes)))
    plot_confusion(s["frame_confusion"], "Frames", ax[0])
    plot_confusion(s["event_confusion"], "Events", ax[1])
    plt.tight_layout(); plt.show()

Events in detail (sorted by confidence in the predicted class). Misclassified events are worth listening to.

In [ ]:
if RUN_CROSS_VALIDATION:
    ev = s["events"].copy()
    ev["confidence"] = ev[classes].max(axis=1)
    display(ev[["event_id", "true_class", "predicted", "correct", "confidence", "n_frames"]]
            .sort_values(["correct", "confidence"]).style.format({"confidence": "{:.2f}"}))

Training curves. Validation loss well above training loss, or rising, suggests overfitting: try more dropout, fewer epochs, or a smaller `DENSE_SIZE`.

In [ ]:
if RUN_CROSS_VALIDATION:
    plot_history(history); plt.tight_layout(); plt.show()

## Train the final model and export to PAMGuard
Trained on all frames with the same bootstrapping. The zip holds the model, the `delphinID.pdtf` settings PAMGuard reads, and an info file with every parameter and the cross-validation accuracy.

In [ ]:
model, final_history = fit_bootstrapped(frames, classes, "label", GROUP_COL, cnn, training, resampling)
info = {"cnn_params": cnn.to_dict(), "training_params": training.to_dict(), "resampling_params": resampling.to_dict(),
        "training_data": {"frames_csv": str(FRAMES_CSV), "n_frames": len(frames), "group_col": GROUP_COL,
                          "groups_per_class": frames.groupby("label")[GROUP_COL].nunique().to_dict()}}
if RUN_CROSS_VALIDATION:
    info["cross_validation"] = {"frame_accuracy": s["frame_accuracy"], "event_accuracy": s["event_accuracy"],
                                "event_accuracy_by_class": s["event_accuracy_by_class"].to_dict()}
    (out / "cv_summary.json").write_text(json.dumps(info["cross_validation"], indent=1))
zip_path = export_pamguard(model, acoustic, classes, out / MODEL_NAME, sample_rate=sample_rate, info=info)
print("Saved", zip_path.resolve())

## Using the model in PAMGuard
1. Add a **Deep Learning Classifier** module. Set its source to the **Whistle and Moan Detector** (whistle model) or **Click Detector** (click model).
2. Choose **delphinID** as the model type and select the zip.
3. Set the segment length, hop and minimum detection value to those in `pamguard_settings` inside the zip's `speciesclassifiers_info.json` (printed below).

A click model only suits recordings at the sample rate it was trained on, because that sets which FFT bins become its inputs.

In [ ]:
import zipfile
with zipfile.ZipFile(zip_path) as z:
    name = next(n for n in z.namelist() if n.endswith("speciesclassifiers_info.json"))
    print(json.dumps(json.loads(z.read(name))["pamguard_settings"], indent=1))